In [ ]:
%load_ext autoreload
%autoreload 2
%cd /opt/tiger/samantha

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import einsum

class GaussianFourierFeatureTransform(nn.Module):
    """
    An implementation of Gaussian Fourier feature mapping.

    "Fourier Features Let Networks Learn High Frequency Functions in Low Dimensional Domains":
       https://arxiv.org/abs/2006.10739
       https://people.eecs.berkeley.edu/~bmild/fourfeat/index.html

    Given an input of size [batches, num_input_channels, width, height],
    returns a tensor of size [batches, mapping_size*2, width, height].
    """

    def __init__(self, input_channels, mapping_size: int, scale: float = 10):
        super().__init__()

        self._input_channels = input_channels
        self._mapping_size = mapping_size
        self.register_buffer("_B", torch.randn((input_channels, mapping_size // 2)) * scale)

    def forward(self, x):
        x = einsum(x, self._B, "b c t, c f -> b f t")
        x = 2 * torch.pi * x
        return torch.cat((torch.sin(x), torch.cos(x)), dim=1)
    
class Sine(nn.Module):
    def __init__(self, w0 = 1.):
        super().__init__()
        self.w0 = w0

    def forward(self, x):
        return torch.sin(self.w0 * x)

class Siren(nn.Module):
    def __init__(
        self,
        dim_in,
        dim_out,
        w0 = 1.,
        c = 6.,
    ):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(dim_in, dim_out, kernel_size=1),
            Sine(w0),
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
import torch
from IPython.display import display, Audio
from samantha.data.utils import read_audio


y, sr = read_audio("/mnt/bn/janne-research-xl/assets/music/10 No Surprises.mp3", 44100, normalize_loudness=False)
start_sec = 0
duration = 10
y = y[..., sr * start_sec:sr * start_sec + sr * duration].mean(dim=1, keepdim=True)
y = y.to("cuda")


In [ ]:
from tqdm.notebook import tqdm

mapping_size = 64
in_channels = y.shape[1]

fourier_transform = GaussianFourierFeatureTransform(in_channels, mapping_size=mapping_size, scale=1.0)
fourier_transform = fourier_transform.to("cuda")
    
model = nn.Sequential(
    Siren(mapping_size, 256),
    Siren(256, 512),
    Siren(512, 512),
    Siren(512, 512),
    Siren(512, 256),
    Siren(256, in_channels),
    # nn.Tanh(), # acts as soft-clip (prevent distortion)
)
model = model.to("cuda")


n_params = sum([p.numel() for p in model.parameters()])
print("Parameters:", n_params)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# data
fourier_features = fourier_transform(y)
print(f"{x.shape} -> {fourier_features.shape}")

losses = []
for _ in tqdm(range(1000)):
    optimizer.zero_grad()
    pred_audio = model(fourier_features)
    loss = F.mse_loss(pred_audio, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

In [ ]:
display(Audio(pred_audio[0].detach().cpu(), rate=sr))

In [ ]:
import matplotlib.pyplot as plt

plt.plot(losses)
plt.show()